# Exploratory Data Analysis with Pyspark and Spark SQL

The following notebook utilizes New York City taxi data from [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

## Instructions

- Load and explore nyc taxi data from january 0f 2019. The exercises can be executed using pyspark or spark sql (a subset of the questions will be re-answered using the language not chosen for the  main work).
- Load the zone lookup table to answer the questions about the nyc boroughs.  
- Load nyc taxi data from January of 2025 and compare data.  
- With any remaining time, work on the where to go from here section.
- Note: the initial lab is opened as read only. To save work completed utilize the `save notebook as` option and give the lab a new name.

In [1]:
import requests

# start a spark session and create a spark context
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("nyc_taxi") \
    .getOrCreate()

sc = spark.sparkContext

In [2]:
# set dl url for January 2019 trip data
download_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-01.parquet'

# get the data
response = requests.get(download_url)

# check that response was good and save the data
jan_2019_trip_data = "yellow_tripdata_2019-01.parquet"
if response.status_code == 200:
    # Changed 'response' to 'data' here
    with open(jan_2019_trip_data, "wb") as f:
        f.write(response.content)


In [3]:
# create the dataframe
df_trips = spark.read.parquet(jan_2019_trip_data)

# A brief note on handling data sources in spark

The command above works well for loading data from parquet files because parquet is a self descibing file format, meaning that the metadata needed to build the dataframe is included directly in the format. However, when working with other formats such as csv or json, a schema must be provided or infered. In production code the schema should always be explicitly provided but during the data exploration phase it is acceptable to infer the schema, and when infering the schema it often best to use `.option("samplingRatio", <small-portion-of-data>)` to avoid using the entire dataset for schema inference.

```python
df_trips = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", ",") \
    .option("samplingRatio", 0.01) \
    .load("large_dataset.csv")
```

In [4]:
# Show the dataframe
df_trips.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|                 N|         151|         239|           1|        7.0|  0.5|    0.5|      1.6

## Lab

### Part 1

This section can be completed either using pyspark commands or sql commands ( There will be a section after in which a self-chosen subset of the questions are re-answered using the language not used for the main section. i.e. if pyspark is chosen for the main lab, sql should be used to repeat some of the questions. )

- Add a column that creates a unique key to identify each record in order to answer questions about individual trips
- Which trip has the highest passanger count
- What is the Average passanger count
- Shortest/longest trip by distance? by time?.
- busiest day/slowest single day
- busiest/slowest time of day ( you may want to bucket these by hour or create timess such as morning, afternoon, evening, late night )
- On average which day of the week is slowest/busiest
- Does trip distance or num passangers affect tip amount
- What was the highest "extra" charge and which trip
- Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?

In [5]:
from pyspark.sql import functions as fct

In [6]:
# Add column that creates a unique key to identify each record

df_trips = df_trips.withColumn(
    "TripID", #name of the column
    fct.monotonically_increasing_id() # how the column is create: by generating id by adding 1 to each line
)

df_trips.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|     TripID|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|                 N|         151|         239|           1

In [7]:
# Which trip has the highest passager count?

df_trips.orderBy("passenger_count", ascending=False) \
    .select("TripID", "passenger_count") \
    .show(1)

# we order the rows by passenger_count with highest on top
# we want to return TripID and passenger_count
# we show only the first row, so the highest count

# OR

# we aggregate the dataset on passenger_count, using maximum function, do it just returns the max value of passenger_count

df_trips.agg({"passenger_count": "max"}).show()

+-----------+---------------+
|     TripID|passenger_count|
+-----------+---------------+
|42950622916|            9.0|
+-----------+---------------+
only showing top 1 row
+--------------------+
|max(passenger_count)|
+--------------------+
|                 9.0|
+--------------------+



In [8]:
# What is the average passanger count?

df_trips.agg({"passenger_count": "avg"}).show()

# same as highest, we aggregade the dataset on passenger_count, using average function, do it just returns the average value of passenger_count

+--------------------+
|avg(passenger_count)|
+--------------------+
|  1.5670317144945614|
+--------------------+



In [9]:
# Shortest/longest trip by distance? by time?

# Longest trip by distance
df_trips.agg({"trip_distance": "max"}).show(1)

# Shortest trip by distance
df_trips.agg({"trip_distance": "min"}).show(1)


# To calculate the same things by time, we need to add a column "trip_duration"
df_trips = df_trips.withColumn(
    "trip_duration", #name of the column
    fct.unix_timestamp("tpep_dropoff_datetime") - fct.unix_timestamp("tpep_pickup_datetime")
    # how the column is create: drop-off time - pick-up time (both convert from string to stamp)
)

# Longest trip by time (in seconds)
df_trips.agg({"trip_duration": "max"}).show(1)

# Shortest trip by time (in seconds)
df_trips.agg({"trip_duration": "min"}).show(1)

+------------------+
|max(trip_distance)|
+------------------+
|             831.8|
+------------------+

+------------------+
|min(trip_distance)|
+------------------+
|               0.0|
+------------------+

+------------------+
|max(trip_duration)|
+------------------+
|           2618881|
+------------------+

+------------------+
|min(trip_duration)|
+------------------+
|          -5056830|
+------------------+



In [10]:
# Busiest/Slowest single day

# To calculate the number of trip each day, we need to add a column "trip_date" (convert without hours, minuts and seconds)
df_trips = df_trips.withColumn(
    "trip_date", #name of the column
    fct.to_date("tpep_pickup_datetime")
    # how the column is create: pick-up time (convert from string to stamp, but only year/month/day)
)

# Busiest day
df_trips.groupBy("trip_date") \
    .agg(fct.count("*").alias("trip_count")) \
    .orderBy("trip_count", ascending=False) \
    .show(1)

# Slowest day
df_trips.groupBy("trip_date") \
    .agg(fct.count("*").alias("trip_count")) \
    .orderBy("trip_count", ascending=True) \
    .show(1)

+----------+----------+
| trip_date|trip_count|
+----------+----------+
|2019-01-25|    292499|
+----------+----------+
only showing top 1 row
+----------+----------+
| trip_date|trip_count|
+----------+----------+
|2019-05-20|         1|
+----------+----------+
only showing top 1 row


In [11]:
# Busiest/Slowest time of day (you may want to bucket these by hour or create timess such as morning, afternoon, evening, late night)

# We need to add a column "time_of_day" (convert with hours only)
df_trips = df_trips.withColumn(
    "time_of_day", #name of the column
    fct.when(fct.hour("tpep_pickup_datetime") < 6, "Late night") # from 0 to 5 = late night
        .when(fct.hour("tpep_pickup_datetime") < 12, "Morning") # from 6 to 11 = morning
        .when(fct.hour("tpep_pickup_datetime") < 18, "Afternoon") # from 12 to 17 = afternoon
        .when(fct.hour("tpep_pickup_datetime") < 22, "Evening") # from 18 to 21 = evening
        .otherwise("Late night") # from 22 to infinite (23) = late night
)

# Busiest time of day
df_trips.groupBy("time_of_day") \
    .agg(fct.count("*").alias("time_count")) \
    .orderBy("time_count", ascending=False) \
    .show(1)

# Slowest time of day
df_trips.groupBy("time_of_day") \
    .agg(fct.count("*").alias("time_count")) \
    .orderBy("time_count", ascending=True) \
    .show(1)

+-----------+----------+
|time_of_day|time_count|
+-----------+----------+
|  Afternoon|   2580478|
+-----------+----------+
only showing top 1 row
+-----------+----------+
|time_of_day|time_count|
+-----------+----------+
| Late night|   1332542|
+-----------+----------+
only showing top 1 row


In [12]:
# On average which day of the week is slowest/busiest

# We need to add a column "day_of_week" (convert with hours only)
df_trips = df_trips.withColumn(
    "day_of_week", #name of the column
    fct.date_format("tpep_pickup_datetime", "EEEE")
    # how the column is create: pick-up time (convert from string to weekday)
)

# Busiest weekday
df_trips.groupBy("day_of_week") \
    .agg(fct.count("*").alias("trip_count")) \
    .orderBy("trip_count", ascending=False) \
    .show(1)

# Lowest weekday
df_trips.groupBy("day_of_week") \
    .agg(fct.count("*").alias("trip_count")) \
    .orderBy("trip_count", ascending=True) \
    .show(1)

+-----------+----------+
|day_of_week|trip_count|
+-----------+----------+
|   Thursday|   1357043|
+-----------+----------+
only showing top 1 row
+-----------+----------+
|day_of_week|trip_count|
+-----------+----------+
|     Sunday|    859905|
+-----------+----------+
only showing top 1 row


In [13]:
# Does trip distance or num passangers affect tip amount?

df_trips.groupBy("passenger_count") \
    .agg(fct.avg("tip_amount").alias("avg_tip_amount")) \
    .orderBy("passenger_count") \
    .show()

+---------------+--------------------+
|passenger_count|      avg_tip_amount|
+---------------+--------------------+
|           NULL|0.061789899553571406|
|            0.0|  1.7869007761051638|
|            1.0|  1.8283524429075058|
|            2.0|  1.8339324029045228|
|            3.0|  1.7955889568213272|
|            4.0|  1.7027097823846875|
|            5.0|  1.8698681146978595|
|            6.0|  1.8568302035247934|
|            7.0|   6.542631578947368|
|            8.0|   6.480689655172414|
|            9.0|  3.1166666666666667|
+---------------+--------------------+



In [14]:
# What was the highest "extra" charge and which trip?

df_trips.orderBy("extra", ascending=False) \
    .show(1)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+------+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+-------------+----------+-----------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount| extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|     TripID|trip_duration| trip_date|time_of_day|day_of_week|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+------+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+-------------+----------+-----------+-----------+
|

Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?

Yes, there are outliers datapoints:
- negative trip duration
- zero distance trip
- zero passenger trip

There are also strange datapoints (but still possible):
- extreme distance trip (up to 831.8 miles)
- extreme passenger count (up to 9 passengers)

### Part 2

- Using the code for loading the first dataset as an example, load in the taxi zone lookup and answer the following questions
- which borough had most pickups? dropoffs?
- what are the busy/slow times by borough 
- what are the busiest days of the week by borough?
- what is the average trip distance by borough?
- what is the average trip fare by borough?
- highest/lowest faire amounts for a trip, what burough is associated with the each
- load the dataset from the most recently available january, is there a change to any of the average metrics.

In [15]:
# Using the code for loading the first dataset as an example, load in the taxi zone lookup and answer the following questions

download_url = 'https://d37ci6vzurychx.cloudfront.net/misc/taxi+_zone_lookup.csv'

# get the data
response = requests.get(download_url)

# check that response was good and save the data
taxi_zone_lookup = "taxi+_zone_lookup.csv"
if response.status_code == 200:
    # Changed 'response' to 'data' here
    with open(taxi_zone_lookup, "wb") as f:
        f.write(response.content)

# create the dataframe
df_zones = spark.read.option("header", "true").option("skipRows", 1).csv(taxi_zone_lookup)
df_zones.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [16]:
# Which borough had most pickups? dropoffs?

# We need to join both dataframes by the LocationID (pick-up)
# we group by borough, count the number of pick-up, and order them by descending value
df_trips.join(df_zones, df_trips.PULocationID == df_zones.LocationID, "inner") \
    .groupBy("Borough") \
    .agg(fct.count("*").alias("pickup_count")) \
    .orderBy("pickup_count", ascending=False) \
    .show(1)

# We need to join both dataframes by the LocationID (drop-off)
# we group by borough, count the number of drop-off, and order them by descending value
df_trips.join(df_zones, df_trips.DOLocationID == df_zones.LocationID, "inner") \
    .groupBy("Borough") \
    .agg(fct.count("*").alias("dropoff_count")) \
    .orderBy("dropoff_count", ascending=False) \
    .show(1)

+---------+------------+
|  Borough|pickup_count|
+---------+------------+
|Manhattan|     6950965|
+---------+------------+
only showing top 1 row
+---------+-------------+
|  Borough|dropoff_count|
+---------+-------------+
|Manhattan|      6817355|
+---------+-------------+
only showing top 1 row


In [17]:
# What are the busy/slow times by borough?

# We start by joining both dataframes, group by borough and time of day
# we count the number of rows, group them by borough
# then we search for the max count per time of day
# and we order all the borough by the max count
df_trips.join(df_zones, df_trips.PULocationID == df_zones.LocationID, "inner") \
    .groupBy("Borough", "time_of_day") \
    .count() \
    .groupBy("Borough") \
    .agg(fct.max(fct.struct("count", "time_of_day")).alias("max")) \
    .select("Borough", "max.time_of_day", "max.count") \
    .orderBy(fct.col("max.count").desc()) \
    .show()

# same way of doing for slow times
df_trips.join(df_zones, df_trips.PULocationID == df_zones.LocationID, "inner") \
    .groupBy("Borough", "time_of_day") \
    .count() \
    .groupBy("Borough") \
    .agg(fct.min(fct.struct("count", "time_of_day")).alias("min")) \
    .select("Borough", "min.time_of_day", "min.count") \
    .orderBy(fct.col("min.count").asc()) \
    .show()


+-------------+-----------+-------+
|      Borough|time_of_day|  count|
+-------------+-----------+-------+
|    Manhattan|  Afternoon|2338785|
|       Queens|  Afternoon| 157388|
|      Unknown|  Afternoon|  54859|
|     Brooklyn|    Morning|  29426|
|        Bronx|    Morning|   7671|
|          N/A| Late night|   1185|
|          EWR|  Afternoon|    248|
|Staten Island|    Morning|    150|
+-------------+-----------+-------+

+-------------+-----------+-------+
|      Borough|time_of_day|  count|
+-------------+-----------+-------+
|          EWR| Late night|     37|
|Staten Island|    Evening|     40|
|          N/A|    Evening|    743|
|        Bronx|    Evening|   2108|
|     Brooklyn|    Evening|  15331|
|      Unknown| Late night|  27504|
|       Queens| Late night|  92608|
|    Manhattan| Late night|1183747|
+-------------+-----------+-------+



In [18]:
# What are the busiest days of the week by borough?

# same way of doing as for time of day
df_trips.join(df_zones, df_trips.PULocationID == df_zones.LocationID, "inner") \
    .groupBy("Borough", "day_of_week") \
    .count() \
    .groupBy("Borough") \
    .agg(fct.max(fct.struct("count", "day_of_week")).alias("max")) \
    .select("Borough", "max.day_of_week", "max.count") \
    .orderBy(fct.col("max.count").desc()) \
    .show()

+-------------+-----------+-------+
|      Borough|day_of_week|  count|
+-------------+-----------+-------+
|    Manhattan|   Thursday|1229554|
|       Queens|   Thursday|  78972|
|      Unknown|   Thursday|  28929|
|     Brooklyn|    Tuesday|  15779|
|        Bronx|   Thursday|   3121|
|          N/A|    Tuesday|    703|
|          EWR|  Wednesday|     83|
|Staten Island|     Friday|     64|
+-------------+-----------+-------+



In [19]:
# What is the average trip distance by borough?

# as always, we start by join both dataframes, and group them by borough
# then we calculate the average trip distance, and order them descending
df_trips.join(df_zones, df_trips.PULocationID == df_zones.LocationID, "inner") \
    .groupBy("Borough") \
    .agg(fct.avg("trip_distance").alias("avg_trip_distance")) \
    .orderBy(fct.col("avg_trip_distance").desc()) \
    .show()

+-------------+------------------+
|      Borough| avg_trip_distance|
+-------------+------------------+
|Staten Island|12.503601108033246|
|       Queens|11.283218499361993|
|        Bronx| 7.233194552098303|
|     Brooklyn| 4.787677275447492|
|          N/A| 3.193850899742941|
|          EWR| 2.641098654708519|
|      Unknown| 2.415464130400774|
|    Manhattan|2.2286693358402596|
+-------------+------------------+



In [20]:
# What is the average trip fare by borough?

# we join both dataframe, group them by borough
# then we calculate the average of fare_amount, and order them descending
df_trips.join(df_zones, df_trips.PULocationID == df_zones.LocationID, "inner") \
    .groupBy("Borough") \
    .agg(fct.avg("fare_amount").alias("avg_fare_amount")) \
    .orderBy(fct.col("avg_fare_amount").desc()) \
    .show()

+-------------+------------------+
|      Borough|   avg_fare_amount|
+-------------+------------------+
|          EWR| 76.24024663677126|
|          N/A|  59.5731593830335|
|Staten Island|45.289861495844896|
|       Queens| 35.14462651722029|
|        Bronx| 26.26890543682963|
|     Brooklyn|18.649132800172286|
|      Unknown|14.944423051653523|
|    Manhattan|10.792468572351568|
+-------------+------------------+



In [21]:
# Highest/lowest fare amounts for a trip, what borough is associated with each?

df_trips.join(df_zones, df_trips.PULocationID == df_zones.LocationID, "inner") \
    .orderBy(fct.col("fare_amount").desc()) \
    .select("fare_amount", "Borough") \
    .show(1)

df_trips.join(df_zones, df_trips.PULocationID == df_zones.LocationID, "inner") \
    .orderBy(fct.col("fare_amount").asc()) \
    .select("fare_amount", "Borough") \
    .show(1)

+-----------+---------+
|fare_amount|  Borough|
+-----------+---------+
|  623259.86|Manhattan|
+-----------+---------+
only showing top 1 row
+-----------+-------+
|fare_amount|Borough|
+-----------+-------+
|     -362.0| Queens|
+-----------+-------+
only showing top 1 row


In [22]:
# Load the dataset from the most recently available january, is there a change to any of the average metrics.

download_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet'

# get the data
response = requests.get(download_url)

# check that response was good and save the data
jan_2025_trip_data = "yellow_tripdata_2025-01.parquet"
if response.status_code == 200:
    # Changed 'response' to 'data' here
    with open(jan_2025_trip_data, "wb") as f:
        f.write(response.content)

# create the dataframe
df_trips_2025 = spark.read.parquet(jan_2025_trip_data)
df_trips_2025.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       1| 2025-01-01 00:18:38|  2025-01-01 00:26:59|              1|          1.6|         1|                 N|         229|    

In [23]:
print("2019 Metrics:")
df_trips.agg(
    fct.avg("trip_distance"),
    fct.avg("fare_amount"),
    fct.avg("passenger_count")
).show()

print("2025 Metrics:")
df_trips_2025.agg(
    fct.avg("trip_distance"),
    fct.avg("fare_amount"),
    fct.avg("passenger_count")
).show()

2019 Metrics:
+------------------+-----------------+--------------------+
|avg(trip_distance)| avg(fare_amount)|avg(passenger_count)|
+------------------+-----------------+--------------------+
|2.8301461681153532|12.52967677747685|  1.5670317144945614|
+------------------+-----------------+--------------------+

2025 Metrics:
+------------------+-----------------+--------------------+
|avg(trip_distance)| avg(fare_amount)|avg(passenger_count)|
+------------------+-----------------+--------------------+
| 5.855126178843539|17.08180276045484|  1.2978589658806226|
+------------------+-----------------+--------------------+



### Part 3

Choose 3 questions from above and re-answer them using the language you did not use for the main notebook . (i.e - if you completed the exercise in python, redo 3 questions in pure sql) . at least one of the questions to be redone must involve a join

In [ ]:
# Reset dataframe
download_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-01.parquet'

# get the data
response = requests.get(download_url)

# check that response was good and save the data
if response.status_code == 200:
    # Changed 'response' to 'data' here
    with open(jan_2019_trip_data, "wb") as f:
        f.write(response.content)

# re-create the dataframe
df_trips = spark.read.parquet(jan_2019_trip_data)

# Replace dataframes by SQL Views
df_trips.createOrReplaceTempView("trips")

In [ ]:
# Which trip has the highest passenger count?

spark.sql("""
    SELECT MAX(passenger_count) AS max_passenger_count
    FROM trips
""").show()

In [ ]:
# What is the average passenger count?

spark.sql("""
    SELECT AVG(passenger_count) AS avg_passenger_count
    FROM trips
""").show()

In [ ]:
# Shortest/Longest trip by distance? by time?

# Longest trip by distance
spark.sql("""
    SELECT MAX(trip_distance) AS max_trip_distance
    FROM trips
""").show()

# Shortest trip by distance
spark.sql("""
    SELECT MIN(trip_distance) AS min_trip_distance
    FROM trips
""").show()

# Longest trip by time
spark.sql("""
    SELECT MAX(unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime))
    AS max_duration_sec
    FROM trips
""").show()

# Shortest trip by time
spark.sql("""
    SELECT MIN(unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime))
    AS min_duration_sec
    FROM trips
""").show()

# Where to go from here

- Continue building the dataset by loading in more data, start by completing the data for 2019 and calculating the busiest season (fall, winter, spring, summer)
- As of spark v4 dataframes have native visualization support. Choose at least 3 questions from above and provide visualizations.
- Explore a dataset/datasets of your choosing